# Notebook 04 — SHAP Explainability Analysis

**Author:** Nandan Kumar K N  
**Project:** AI-Driven Pharmacogenomics — Asian ethnic subgroups  
**Goal:** Apply SHAP to the best XGBoost models. Identify which variants drive predictions differently across SAS and EAS subgroups. Produce Figure 6 (SHAP summary) and Figure 7 (cross-population feature comparison).

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb
import shap
import joblib
import json

ROOT      = Path(r"D:\GIT\asian-pgx-ml")
PROC_DIR  = ROOT / 'data' / 'processed'
MODEL_DIR = ROOT / 'results' / 'models'
FIG_DIR   = ROOT / 'results' / 'figures'
TAB_DIR   = ROOT / 'results' / 'tables'

POPS = ['GIH', 'ITU', 'BEB', 'CHB', 'CHS', 'JPT']
SAS  = ['GIH', 'ITU', 'BEB']
EAS  = ['CHB', 'CHS', 'JPT']

print("✓ Imports OK")
print(f"  SHAP version: {shap.__version__}")


## 1. Load feature matrix and rebuild labels


In [ ]:
fm = pd.read_csv(PROC_DIR / 'feature_matrix_full.csv', index_col=0)

# ── Rebuild phenotype labels (same logic as notebook 03) ──────────────────
star2_col = 'CYP2C19_10:94842865:C>T'
fm['CYP2C19_PM_binary'] = (fm[star2_col].apply(
    pd.to_numeric, args=('coerce',)).fillna(0) >= 2).astype(int)

cyp2d6_cols = [c for c in fm.columns
               if c.startswith('CYP2D6_') and '_TPM_' not in c]
fm[cyp2d6_cols] = fm[cyp2d6_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
fm['CYP2D6_burden'] = fm[cyp2d6_cols].sum(axis=1)
b90 = fm['CYP2D6_burden'].quantile(0.90)
fm['CYP2D6_PM_binary'] = (fm['CYP2D6_burden'] >= b90).astype(int)

META_COLS  = ['population', 'super_population',
              'CYP2C19_PM_binary', 'CYP2D6_PM_binary', 'CYP2D6_burden']
LABEL_SNPS = ['CYP2C19_10:94842865:C>T', 'CYP2C19_10:94781859:G>A']
FEATURE_COLS = [c for c in fm.columns
                if c not in META_COLS and c not in LABEL_SNPS]
fm[FEATURE_COLS] = fm[FEATURE_COLS].apply(pd.to_numeric, errors='coerce').fillna(0)

print(f"Feature matrix: {fm.shape}")
print(f"Features: {len(FEATURE_COLS)}")
print(f"CYP2D6 PM count: {fm['CYP2D6_PM_binary'].sum()} / {len(fm)}")


## 2. Train XGBoost models per cohort for SHAP


In [ ]:
# Train XGBoost on full data per cohort (pooled, SAS, EAS)
# Primary target: CYP2D6 (realistic AUC, burden-based labels)

def train_xgb(df, feature_cols, target_col):
    X = df[feature_cols].fillna(0)
    y = df[target_col]
    pos, neg = int(y.sum()), int(len(y) - y.sum())
    model = xgb.XGBClassifier(
        learning_rate=0.05, n_estimators=300, max_depth=6,
        scale_pos_weight=neg/pos if pos > 0 else 1,
        eval_metric='logloss', random_state=42, verbosity=0
    )
    model.fit(X, y)
    return model, X, y

print("Training XGBoost models...")

models = {}
X_sets = {}

# Pooled
models['pooled'], X_sets['pooled'], _ = train_xgb(fm, FEATURE_COLS, 'CYP2D6_PM_binary')
print(f"  ✓ Pooled  : {len(fm)} samples")

# SAS
sas_df = fm[fm['population'].isin(SAS)]
models['SAS'], X_sets['SAS'], _ = train_xgb(sas_df, FEATURE_COLS, 'CYP2D6_PM_binary')
print(f"  ✓ SAS     : {len(sas_df)} samples")

# EAS
eas_df = fm[fm['population'].isin(EAS)]
models['EAS'], X_sets['EAS'], _ = train_xgb(eas_df, FEATURE_COLS, 'CYP2D6_PM_binary')
print(f"  ✓ EAS     : {len(eas_df)} samples")

print("\n✓ All models trained")


## 3. Compute SHAP values


In [ ]:
print("Computing SHAP values (this may take 2-3 minutes)...")

shap_values = {}
explainers  = {}

for cohort, model in models.items():
    X = X_sets[cohort]
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X)
    shap_values[cohort] = sv
    explainers[cohort]  = explainer
    print(f"  ✓ {cohort:8s}: SHAP matrix shape {sv.shape}")

print("\n✓ SHAP values computed for all cohorts")


## 4. Top features per cohort


In [ ]:
def get_top_features(shap_vals, feature_cols, n=20):
    """Return top n features by mean absolute SHAP value."""
    mean_abs = np.abs(shap_vals).mean(axis=0)
    top_idx  = np.argsort(mean_abs)[::-1][:n]
    return pd.DataFrame({
        'feature':    [feature_cols[i] for i in top_idx],
        'mean_abs_shap': mean_abs[top_idx]
    })

top_features = {}
for cohort in ['pooled', 'SAS', 'EAS']:
    top_features[cohort] = get_top_features(
        shap_values[cohort], FEATURE_COLS, n=20
    )
    print(f"\nTop 10 features — {cohort}:")
    print(top_features[cohort].head(10).to_string(index=False))

# Save top features tables
for cohort, df in top_features.items():
    df.to_csv(TAB_DIR / f'table4_shap_top_features_{cohort}.csv', index=False)
print("\n✓ SHAP feature tables saved")


## 5. Figure 6 — SHAP summary plots


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
cohort_titles = {'pooled': 'Pooled (all 610)', 'SAS': 'South Asian (SAS)', 'EAS': 'East Asian (EAS)'}
cohort_colors = {'pooled': '#1A5276', 'SAS': '#1565C0', 'EAS': '#C62828'}

for ax_idx, cohort in enumerate(['pooled', 'SAS', 'EAS']):
    ax = axes[ax_idx]
    top = top_features[cohort].head(15)

    # Clean feature names for display
    def clean_name(f):
        f = f.replace('CYP2D6_', '').replace('CYP2C19_', 'C19_')
        f = f.replace('CYP3A5_', 'C3A5_').replace('NUDT15_', 'N15_')
        f = f.replace('SLCO1B1_', 'SLCO_')
        if '_TPM_' in f:
            gene = f.split('_TPM_')[0]
            metric = f.split('_TPM_')[1]
            return f'{gene} TPM ({metric})'
        return f[:30]

    names  = [clean_name(f) for f in top['feature']]
    values = top['mean_abs_shap'].values

    bars = ax.barh(range(len(names)), values,
                    color=cohort_colors[cohort], alpha=0.8,
                    edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Mean |SHAP value|', fontsize=9)
    ax.set_title(f'{cohort_titles[cohort]}\nTop 15 features',
                  fontweight='bold', fontsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Figure 6: SHAP Feature Importance — CYP2D6 PM Prediction\n'
             'Comparing pooled vs population-stratified models',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'figure6_shap_summary.png', dpi=300,
            bbox_inches='tight', facecolor='white')
print("✓ Figure 6 saved")
plt.show()


## 6. Figure 7 — SAS vs EAS feature divergence


In [ ]:
# Compare top feature rankings between SAS and EAS
# Features that rank high in one but not the other = population-specific drivers

sas_top = set(top_features['SAS']['feature'].head(20))
eas_top = set(top_features['EAS']['feature'].head(20))

shared      = sas_top & eas_top
sas_only    = sas_top - eas_top
eas_only    = eas_top - sas_top

print("Feature overlap analysis (top 20 features):")
print(f"  Shared between SAS and EAS : {len(shared)}")
print(f"  SAS-specific (not in EAS)  : {len(sas_only)}")
print(f"  EAS-specific (not in SAS)  : {len(eas_only)}")
print(f"\nSAS-specific top features:")
for f in list(sas_only)[:5]:
    rank = top_features['SAS'][top_features['SAS']['feature']==f].index[0] + 1
    print(f"  Rank {rank}: {f}")
print(f"\nEAS-specific top features:")
for f in list(eas_only)[:5]:
    rank = top_features['EAS'][top_features['EAS']['feature']==f].index[0] + 1
    print(f"  Rank {rank}: {f}")


In [ ]:
# ── Dot plot: SAS rank vs EAS rank for top 30 features ───────────────────
all_top = list(set(
    top_features['SAS']['feature'].head(30).tolist() +
    top_features['EAS']['feature'].head(30).tolist()
))

sas_rank = {f: i+1 for i, f in enumerate(top_features['SAS']['feature'])}
eas_rank = {f: i+1 for i, f in enumerate(top_features['EAS']['feature'])}

plot_data = []
for f in all_top:
    sr = sas_rank.get(f, 35)
    er = eas_rank.get(f, 35)
    plot_data.append({'feature': f, 'SAS_rank': sr, 'EAS_rank': er,
                       'rank_diff': abs(sr - er)})

plot_df = pd.DataFrame(plot_data).sort_values('rank_diff', ascending=False)

fig, ax = plt.subplots(figsize=(8, 8))

scatter = ax.scatter(
    plot_df['SAS_rank'], plot_df['EAS_rank'],
    c=plot_df['rank_diff'], cmap='RdYlBu_r',
    s=80, alpha=0.8, edgecolors='white', linewidth=0.5
)

# Diagonal line (perfect agreement)
ax.plot([1, 35], [1, 35], 'k--', linewidth=1, alpha=0.4, label='Perfect agreement')

# Label most divergent features
for _, row in plot_df.head(8).iterrows():
    name = row['feature'].replace('CYP2D6_', '').replace('CYP2C19_', 'C19_')[:20]
    ax.annotate(name, (row['SAS_rank'], row['EAS_rank']),
                fontsize=7, ha='left', va='bottom',
                xytext=(3, 3), textcoords='offset points')

plt.colorbar(scatter, ax=ax, label='|SAS rank − EAS rank|')
ax.set_xlabel('Feature rank in SAS model', fontsize=10)
ax.set_ylabel('Feature rank in EAS model', fontsize=10)
ax.set_title('Figure 7: Population-Specific Feature Divergence\n'
              'SAS vs EAS SHAP feature rankings (CYP2D6 PM)',
              fontweight='bold', fontsize=11)
ax.set_xlim(0, 36)
ax.set_ylim(0, 36)
ax.invert_xaxis()
ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(FIG_DIR / 'figure7_shap_sas_eas_divergence.png', dpi=300,
            bbox_inches='tight', facecolor='white')
print("✓ Figure 7 saved")
plt.show()


## 7. SHAP waterfall for one individual


In [ ]:
# ── Show SHAP explanation for a single predicted PM individual ────────────
# Pick the individual with highest PM probability in the pooled model
X_pooled = X_sets['pooled']
probs = models['pooled'].predict_proba(X_pooled)[:,1]
top_pm_idx = np.argmax(probs)

print(f"Highest PM probability individual:")
print(f"  Index    : {top_pm_idx}")
print(f"  PM prob  : {probs[top_pm_idx]:.4f}")
print(f"  Population: {fm.iloc[top_pm_idx]['population']}")

# SHAP explanation for this individual
sv_single = shap_values['pooled'][top_pm_idx]
top_n = 10
top_feat_idx = np.argsort(np.abs(sv_single))[::-1][:top_n]

fig, ax = plt.subplots(figsize=(9, 5))
feat_names  = [FEATURE_COLS[i].replace('CYP2D6_','').replace('CYP2C19_','C19_')[:25]
               for i in top_feat_idx]
shap_vals_s = sv_single[top_feat_idx]
colors = ['#C0392B' if v > 0 else '#2471A3' for v in shap_vals_s]

bars = ax.barh(range(top_n), shap_vals_s, color=colors,
                edgecolor='white', linewidth=0.5)
ax.set_yticks(range(top_n))
ax.set_yticklabels(feat_names, fontsize=9)
ax.invert_yaxis()
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('SHAP value (impact on PM prediction)', fontsize=10)
ax.set_title(f'Figure 8: SHAP Explanation — Single Individual\n'
              f'Population: {fm.iloc[top_pm_idx]["population"]} | '
              f'PM probability: {probs[top_pm_idx]:.3f}',
              fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#C0392B', label='Increases PM risk'),
                   Patch(facecolor='#2471A3', label='Decreases PM risk')]
ax.legend(handles=legend_elements, loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / 'figure8_shap_individual.png', dpi=300,
            bbox_inches='tight', facecolor='white')
print("✓ Figure 8 saved")
plt.show()


In [ ]:
print("="*60)
print("NOTEBOOK 04 COMPLETE")
print("="*60)

print(f"\nSHAP analysis summary:")
print(f"  Cohorts analysed    : pooled, SAS, EAS")
print(f"  Top features saved  : results/tables/table4_shap_top_features_*.csv")
print(f"  Shared features (top 20): {len(shared)}/20")
print(f"  SAS-specific        : {len(sas_only)} features")
print(f"  EAS-specific        : {len(eas_only)} features")

print(f"\nFigures saved:")
print(f"  Figure 6 → figure6_shap_summary.png")
print(f"  Figure 7 → figure7_shap_sas_eas_divergence.png")
print(f"  Figure 8 → figure8_shap_individual.png")

print(f"\nKey paper sentence:")
print(f"  SHAP analysis revealed that {len(sas_only)} of the top 20 predictive")
print(f"  features were unique to the SAS model and {len(eas_only)} unique to")
print(f"  the EAS model, demonstrating that population-specific variant")
print(f"  combinations drive CYP2D6 phenotype prediction differently")
print(f"  across South Asian and East Asian subgroups.")

print(f"\nNext: notebook 05 — CPIC clinical dosing mapping")
